In [19]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
    size_adjusted_power_comparison,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = False
_AUGMENTED_PARAM = 'x_coef'
_AUGMENTED_EQUATION = 'OutGap'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.025
_MC_SAMPLES = 1000
_MC_ALPHA = 0.05
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)


In [20]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83   0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [21]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [22]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [23]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [24]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: False
Augmented measurement equation: OutGap
Augmented coefficient: x_coef
Monte Carlo replications: 1000
Noise Covariance:
 [[0.302 0.    0.   ]
 [0.    0.42  0.   ]
 [0.    0.    0.019]]


In [25]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 1000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,16.384,0.003,0.214,0.001,1000,987,0.987,0.004,0.978,0.992
1,Infl,24.848,0.000,0.252,0.000,1000,1000,1.000,0.000,0.996,1.000
2,Rate,9.719,0.033,0.178,0.003,1000,869,0.869,0.011,0.847,0.889


In [26]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 1000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.124,2.593,0.563,0.002,0.076,0.009,1000,40,0.04,0.006,0.030,0.054,3.0,200,4
1,cov_identity,15.154,433.772,0.000,0.056,4.759,0.000,1000,1000,1.00,0.000,0.996,1.000,6.0,200,4


In [27]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,-2.119,-0.088,1.718,-1.248,0.303,0.012,0.052,0.002,0.005,0.031,0.009,0.0,1000,235,0.235,0.013,0.210,0.262
1,OutGap,x,-0.300,-0.097,0.216,-1.377,0.279,0.014,0.007,0.002,0.001,0.030,0.009,0.0,1000,264,0.264,0.014,0.238,0.292
2,OutGap,r,-0.483,-0.015,2.012,-0.218,0.499,0.005,0.063,0.002,0.007,0.031,0.009,0.0,1000,50,0.050,0.007,0.038,0.065
3,Infl,Pi,-0.675,-0.023,1.969,-0.320,0.480,0.006,0.064,0.002,0.006,0.032,0.009,0.0,1000,66,0.066,0.008,0.052,0.083
4,Infl,x,0.053,0.015,0.248,0.209,0.489,0.005,0.008,0.002,0.001,0.033,0.009,0.0,1000,60,0.060,0.008,0.047,0.076
5,Infl,r,-0.393,-0.011,2.298,-0.149,0.483,0.005,0.076,0.002,0.008,0.033,0.009,0.0,1000,53,0.053,0.007,0.041,0.069
6,Rate,Pi,-0.103,-0.021,0.370,-0.292,0.495,0.005,0.012,0.002,0.001,0.031,0.009,0.0,1000,59,0.059,0.007,0.046,0.075
7,Rate,x,0.017,0.025,0.047,0.359,0.485,0.006,0.001,0.002,0.000,0.032,0.009,0.0,1000,62,0.062,0.008,0.049,0.079
8,Rate,r,-0.317,-0.050,0.432,-0.714,0.432,0.007,0.014,0.002,0.002,0.032,0.010,0.0,1000,116,0.116,0.010,0.098,0.137


In [28]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-4.076,-0.327,0.829,-4.900,0.000,0.110,0.026,0.002,0.001,0.028,0.000,0.001,1000,999,0.999,0.001,0.994,1.000
1,OutGap,x,-0.503,-0.328,0.102,-4.910,0.000,0.110,0.003,0.002,0.000,0.029,0.000,0.001,1000,999,0.999,0.001,0.994,1.000
0,OutGap,r,1.491,0.055,1.902,0.775,0.459,0.005,0.044,0.002,0.007,0.022,0.009,0.000,1000,51,0.051,0.007,0.039,0.066
5,Infl,Pi,-0.159,-0.010,1.002,-0.139,0.494,0.005,0.032,0.002,0.001,0.032,0.009,0.000,1000,49,0.049,0.007,0.037,0.064
4,Infl,x,-0.002,0.000,0.123,0.004,0.506,0.005,0.004,0.002,0.000,0.032,0.009,0.000,1000,48,0.048,0.007,0.036,0.063
3,Infl,r,-0.341,-0.010,2.173,-0.144,0.488,0.005,0.071,0.002,0.008,0.033,0.009,0.000,1000,60,0.060,0.008,0.047,0.076
8,Rate,Pi,0.013,0.004,0.189,0.053,0.514,0.005,0.006,0.002,0.000,0.031,0.009,0.000,1000,41,0.041,0.006,0.030,0.055
7,Rate,x,0.008,0.023,0.023,0.333,0.489,0.006,0.001,0.002,0.000,0.032,0.009,0.000,1000,66,0.066,0.008,0.052,0.083
6,Rate,r,-0.343,-0.057,0.408,-0.814,0.412,0.008,0.013,0.002,0.001,0.032,0.010,0.000,1000,127,0.127,0.011,0.108,0.149


In [29]:
print("Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"]).round(3)

Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.684,-3.804,-2.119,-2.119,0.0,0.0,0.0,0.034,0.025,0.052,0.052,0.0,0.0,0.0
1,OutGap,x,0.024,-0.324,-0.300,-0.300,0.0,0.0,0.0,0.004,0.004,0.007,0.007,0.0,0.0,0.0
2,OutGap,r,-0.162,-0.320,-0.483,-0.483,0.0,0.0,0.0,0.039,0.031,0.063,0.063,0.0,0.0,0.0
3,Infl,Pi,-0.025,-0.650,-0.675,-0.675,-0.0,0.0,0.0,0.010,0.063,0.064,0.064,0.0,0.0,0.0
4,Infl,x,0.001,0.051,0.053,0.053,0.0,0.0,0.0,0.001,0.008,0.008,0.008,0.0,0.0,0.0
5,Infl,r,-0.014,-0.378,-0.393,-0.393,-0.0,0.0,0.0,0.012,0.075,0.076,0.076,0.0,0.0,0.0
6,Rate,Pi,0.004,-0.107,-0.103,-0.103,0.0,0.0,0.0,0.002,0.011,0.012,0.012,0.0,0.0,0.0
7,Rate,x,-0.000,0.017,0.017,0.017,-0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
8,Rate,r,-0.006,-0.311,-0.317,-0.317,0.0,0.0,0.0,0.003,0.014,0.014,0.014,0.0,0.0,0.0


In [30]:
print("Innovation decomposition on raw predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"]).round(3)

Innovation decomposition on raw predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.919,-5.995,-4.076,-4.076,0.0,0.0,0.0,0.017,0.011,0.026,0.026,0.0,0.0,0.0
1,OutGap,x,0.208,-0.711,-0.503,-0.503,-0.0,0.0,0.0,0.002,0.002,0.003,0.003,0.0,0.0,0.0
2,OutGap,r,-0.731,2.222,1.491,1.491,-0.0,0.0,0.0,0.047,0.035,0.044,0.044,0.0,0.0,0.0
3,Infl,Pi,-0.011,-0.148,-0.159,-0.159,-0.0,0.0,0.0,0.005,0.032,0.032,0.032,0.0,0.0,0.0
4,Infl,x,-0.001,-0.001,-0.002,-0.002,-0.0,0.0,0.0,0.001,0.004,0.004,0.004,0.0,0.0,0.0
5,Infl,r,-0.011,-0.330,-0.341,-0.341,0.0,0.0,0.0,0.011,0.071,0.071,0.071,0.0,0.0,0.0
6,Rate,Pi,0.003,0.010,0.013,0.013,0.0,0.0,0.0,0.001,0.006,0.006,0.006,0.0,0.0,0.0
7,Rate,x,0.000,0.008,0.008,0.008,0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
8,Rate,r,-0.005,-0.338,-0.343,-0.343,0.0,0.0,0.0,0.002,0.013,0.013,0.013,0.0,0.0,0.0


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.


In [31]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()

## Diagnostics of the Augmented Model

### Marginal LR Test Conditional on $\theta_0$

In [32]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,1.383,-2312.396,-994.693,2635.406,0.0,0.002,5.657,0.631,10.689,0.0,1000,1000,1.0,0.0,0.996,1.0


In [33]:
res_mle

OptimizationResult(kind='mle', x=array([1.25650239]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(0.0), 'x_coef': np.float64(1.2565023864714884), 'r_coef': np.float64(0.0)}, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', fun=np.float64(974.6861593700413), loglik=np.float64(-974.6861593700413), logprior=np.float64(0.0), logpost=np.float64(-974.6861593700413), nfev=16, nit=7, raw=  message: CONVERGENCE: R

## Serial Autocorrelation Tests for the Augmented Model

In [34]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.908,0.370,0.074,0.009,1000,159,0.159,0.012,0.138,0.183
1,Infl,13.573,0.009,0.200,0.001,1000,958,0.958,0.006,0.944,0.969
2,Rate,6.287,0.099,0.149,0.006,1000,618,0.618,0.015,0.587,0.648


In [35]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.124,2.593,0.563,0.002,0.076,0.009,1000,40,0.04,0.006,0.030,0.054,3.0,200,4
1,cov_identity,15.154,433.772,0.000,0.056,4.759,0.000,1000,1000,1.00,0.000,0.996,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.098,2.912,0.516,0.001,0.078,0.009,1000,48,0.048,0.007,0.036,0.063,3.0,200,4
1,cov_identity,0.757,118.227,0.000,0.004,1.742,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


Reference-minus-augmented moment distance comparison:


,test,n_replications,distance_ref,mc_se_distance_ref,distance_aug,mc_se_distance_aug,distance_improvement,mc_se_distance_improvement,stat_ref,mc_se_stat_ref,stat_aug,mc_se_stat_aug,stat_improvement,mc_se_stat_improvement,aug_closer_rate,aug_closer_rate_mc_se,aug_closer_ci_low,aug_closer_ci_high
0,mean_zero_hac,1000,0.124,0.002,0.098,0.001,0.026,0.001,2.593,0.076,2.912,0.078,-0.319,0.024,0.874,0.01,0.852,0.893
1,cov_identity,1000,15.154,0.056,0.757,0.004,14.397,0.056,433.772,4.759,118.227,1.742,315.545,4.518,1.000,0.00,0.996,1.000
